In [1]:
# 1. DATA LOADING & AUDIT
# ----------------------------------------------------

import json
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Environment & Directory Setup
# Data Directory Path 
DATA_DIR = Path("/kaggle/input/datasets/hamaz911/streamintel360-streaming-data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")  # Local fallback directory

# Setting path directory
ARTIFACT_ROOT = Path("/kaggle/working/streamintel360_artifacts") / "notebook_03_recommendations"

FIGURES_DIR = ARTIFACT_ROOT / "figures"
MODELS_DIR = ARTIFACT_ROOT / "models"
METRICS_DIR = ARTIFACT_ROOT / "metrics"
PROCESSED_DIR = ARTIFACT_ROOT / "processed_data"

for folder in [FIGURES_DIR, MODELS_DIR, METRICS_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def save_fig(fig_name: str, tight_layout: bool = True, fig_extension: str = "png", resolution: int = 300):
    import matplotlib.pyplot as plt
    if tight_layout: 
        plt.tight_layout()
        
    path = FIGURES_DIR / f"{fig_name}.{fig_extension}"
    plt.savefig(path, format=fig_extension, dpi=resolution, bbox_inches="tight")
    print(f"Figure saved: {path}")


# Mapping dataset keys to CSV filenames
files = {
    "movies": "movies.csv",
    "users": "users.csv",
    "watch_history": "watch_history.csv",
    "reviews": "reviews.csv",
    "search_logs": "search_logs.csv",
    "recommendation_logs": "recommendation_logs.csv"
}

# Loading datasets into dictionary
dfs = {}
for name, file in files.items():
    path = DATA_DIR / file
    if not path.exists():
        print(f"Missing: {file} at {path}")
        continue
    dfs[name] = pd.read_csv(path, low_memory=False)

# Dataset Summary Table
print("DATASET OVERVIEW & AUDIT SUMMARY")
summary_data = []
for name, df in dfs.items():
    summary_data.append({
        "Dataset": name,
        "Rows": f"{len(df):,}",
        "Columns": len(df.columns),
        "Duplicates": f"{df.duplicated().sum():,}",
        "Missing Values": f"{df.isna().sum().sum():,}"
    })
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Schema Inspection
print("SCHEMA & DATA TYPE AUDIT")
for name, df in dfs.items():
    print(f"\n[{name.upper()}] — Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    missing_cols = df.isna().sum()[df.isna().sum() > 0]
    if missing_cols.empty:
        print("Missing per column: None")
    else:
        print("Missing per column:")
        for col, val in missing_cols.items():
            print(f" -> {col}: {val:,}")

DATASET OVERVIEW & AUDIT SUMMARY
            Dataset    Rows  Columns Duplicates Missing Values
             movies   1,040       18         40          3,671
              users  10,300       16        300          4,615
      watch_history 105,000       12      5,000        104,749
            reviews  15,450       12        450          5,624
        search_logs  26,500       11      1,500         14,826
recommendation_logs  52,000       11      2,000          7,836
SCHEMA & DATA TYPE AUDIT

[MOVIES] — Shape: (1040, 18)
Columns: ['movie_id', 'title', 'content_type', 'genre_primary', 'genre_secondary', 'release_year', 'duration_minutes', 'rating', 'language', 'country_of_origin', 'imdb_rating', 'production_budget', 'box_office_revenue', 'number_of_seasons', 'number_of_episodes', 'is_netflix_original', 'added_to_platform', 'content_warning']
Missing per column:
 -> genre_secondary: 667
 -> imdb_rating: 150
 -> production_budget: 675
 -> box_office_revenue: 709
 -> number_of_seasons: 7

In [2]:
# 2. DATA CLEANING, FEATURE ENGINEERING & SPLIT
# -----------------------------------------------------------

# Cleaning Movies Metadata
movies_clean = dfs["movies"].drop_duplicates("movie_id").copy()
movies_clean["duration_minutes"] = (
    pd.to_numeric(movies_clean["duration_minutes"], errors="coerce").replace(0, np.nan).fillna(90))
movies_clean["genre_primary"] = movies_clean["genre_primary"].fillna("Unknown")

# Cleaning Watch History Records
watch_clean = dfs["watch_history"].drop_duplicates("session_id").copy()
watch_clean["watch_date"] = pd.to_datetime(watch_clean["watch_date"], errors="coerce")
watch_clean = watch_clean.dropna(subset=["user_id", "movie_id", "watch_date"])

# Filtering against valid entities in Users and Movies tables
valid_users = set(dfs["users"]["user_id"])
valid_movies = set(movies_clean["movie_id"])

watch_clean = watch_clean[
    watch_clean["user_id"].isin(valid_users) & watch_clean["movie_id"].isin(valid_movies)].copy()

# Merging Movie Duration to compute Completion Ratio
watch_clean = watch_clean.merge(
    movies_clean[["movie_id", "duration_minutes"]], on="movie_id", how="left"
)

watch_clean["completion_ratio"] = (
    watch_clean["watch_duration_minutes"] / watch_clean["duration_minutes"]).clip(0, 1).fillna(0)

# Mapping Behavioral Action Scores
action_weights = {"completed": 1.0,"watched": 0.8,"paused": 0.4,"abandoned": 0.1}
watch_clean["action_clean"] = watch_clean["action"].astype(str).str.lower().str.strip()
watch_clean["action_score"] = watch_clean["action_clean"].map(action_weights).fillna(0.5)

# Implicit Engagement Index (IEI)
watch_clean["iei_score"] = (
    0.70 * watch_clean["completion_ratio"] + 0.30 * watch_clean["action_score"]).clip(0, 1)

# Temporal Data Split (80% Train / 10% Validation / 10% Test)
watch_clean = watch_clean.sort_values("watch_date").reset_index(drop=True)
n_records = len(watch_clean)
train_end = int(n_records * 0.80)
val_end = int(n_records * 0.90)

train_df = watch_clean.iloc[:train_end].copy()
val_df = watch_clean.iloc[train_end:val_end].copy()
test_df = watch_clean.iloc[val_end:].copy()

# Exporting Processed Data Artifacts
movies_clean.to_csv(PROCESSED_DIR / "movies_clean.csv", index=False)
watch_clean.to_csv(PROCESSED_DIR / "watch_clean.csv", index=False)
train_df.to_csv(PROCESSED_DIR / "train_df.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val_df.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_df.csv", index=False)

# Summary & Audit Output
print("DATA PREPROCESSING & TEMPORAL SPLIT SUMMARY")
print(f"Cleaned Movies Records       : {len(movies_clean):,}")
print(f"Valid Watch Sessions         : {len(watch_clean):,}")
print("\nImplicit Engagement Index (IEI) Descriptive Stats:")
print(watch_clean[["completion_ratio", "action_score", "iei_score"]].describe().round(3).to_string())

print("\nTemporal Data Split Overview:")
print(f"  Train Set      ({len(train_df):>8,} rows) : {train_df['watch_date'].min()} → {train_df['watch_date'].max()}")
print(f"  Validation Set ({len(val_df):>8,} rows) : {val_df['watch_date'].min()} → {val_df['watch_date'].max()}")
print(f"  Test Set       ({len(test_df):>8,} rows) : {test_df['watch_date'].min()} → {test_df['watch_date'].max()}")
print(f"\nProcessed artifacts successfully exported to: {PROCESSED_DIR.resolve()}")

DATA PREPROCESSING & TEMPORAL SPLIT SUMMARY
Cleaned Movies Records       : 1,000
Valid Watch Sessions         : 100,000

Implicit Engagement Index (IEI) Descriptive Stats:
       completion_ratio  action_score   iei_score
count        100000.000    100000.000  100000.000
mean              0.564         0.599       0.575
std               0.372         0.234       0.269
min               0.000         0.400       0.120
25%               0.229         0.500       0.335
50%               0.562         0.500       0.577
75%               1.000         0.500       0.820
max               1.000         1.000       1.000

Temporal Data Split Overview:
  Train Set      (  80,000 rows) : 2024-01-01 00:00:00 → 2025-08-07 00:00:00
  Validation Set (  10,000 rows) : 2025-08-07 00:00:00 → 2025-10-20 00:00:00
  Test Set       (  10,000 rows) : 2025-10-20 00:00:00 → 2025-12-31 00:00:00

Processed artifacts successfully exported to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendatio

In [3]:
# 3. USER–ITEM INTERACTION MATRIX
# ---------------------------------------

# 1Aggregating repeated user-movie interactions via mean IEI score
interaction_df = (
    train_df.groupby(["user_id", "movie_id"], as_index=False)["iei_score"].mean())

print("USER–ITEM INTERACTION MATRIX CONSTRAINTS & SPARSITY AUDIT")
print(f"Unique User-Movie Interaction Pairs : {len(interaction_df):,}")

# Constructing Dense Pivot Matrix
interaction_matrix = interaction_df.pivot(
    index="user_id", columns="movie_id", values="iei_score").fillna(0)
print(f"Interaction Matrix Dimensions       : {interaction_matrix.shape[0]:,} Users × {interaction_matrix.shape[1]:,} Movies")

# Calculating Matrix Sparsity
total_cells = interaction_matrix.shape[0] * interaction_matrix.shape[1]
observed_cells = (interaction_matrix.values > 0).sum()
sparsity = (1.0 - (observed_cells / total_cells)) * 100.0

print(f"Observed Non-Zero Interactions     : {observed_cells:,}")
print(f"Matrix Sparsity Ratio               : {sparsity:.2f}%")

# Converting to Scipy Sparse CSR Matrix for Memory & Compute Efficiency
sparse_interaction_matrix = sp.csr_matrix(interaction_matrix.values)

# Extracting User and Movie ID Index Mappers
user_ids = list(interaction_matrix.index)
movie_ids = list(interaction_matrix.columns)

user2idx = {uid: idx for idx, uid in enumerate(user_ids)}
idx2user = {idx: uid for idx, uid in enumerate(user_ids)}
movie2idx = {mid: idx for idx, mid in enumerate(movie_ids)}
idx2movie = {idx: mid for idx, mid in enumerate(movie_ids)}

# Exporting Matrix Artifacts & ID Mappings
interaction_matrix.to_csv(PROCESSED_DIR / "user_item_matrix.csv")
sp.save_npz(PROCESSED_DIR / "user_item_matrix_sparse.npz", sparse_interaction_matrix)

mappings = {"user2idx": user2idx,"idx2user": idx2user,"movie2idx": movie2idx,"idx2movie": idx2movie}
joblib.dump(mappings, MODELS_DIR / "interaction_matrix_mappings.joblib")

# Descriptive Interaction Distribution Statistics
user_interactions = interaction_df.groupby("user_id").size()
movie_interactions = interaction_df.groupby("movie_id").size()

print("INTERACTION DISTRIBUTION STATISTICS")
print("\nUser Interaction Distribution:")
print(user_interactions.describe().round(2).to_string())
print("\nMovie Interaction Distribution:")
print(movie_interactions.describe().round(2).to_string())

# Mapping Top 10 Most Interacted Movies with Titles
movie_titles = movies_clean.set_index("movie_id")["title"].to_dict()
top_movies_df = (
    movie_interactions.sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={0: "interaction_count"}))
top_movies_df["movie_title"] = top_movies_df["movie_id"].map(movie_titles)
print("\nTop 10 Most-Interacted Movies:")
print(top_movies_df[["movie_id", "movie_title", "interaction_count"]].to_string(index=False))
print(f"\nMatrix artifacts and sparse matrices exported to:\n - {PROCESSED_DIR.resolve()}\n - {MODELS_DIR.resolve()}")

USER–ITEM INTERACTION MATRIX CONSTRAINTS & SPARSITY AUDIT
Unique User-Movie Interaction Pairs : 79,653
Interaction Matrix Dimensions       : 9,997 Users × 1,000 Movies
Observed Non-Zero Interactions     : 79,653
Matrix Sparsity Ratio               : 99.20%
INTERACTION DISTRIBUTION STATISTICS

User Interaction Distribution:
count    9997.00
mean        7.97
std         2.81
min         1.00
25%         6.00
50%         8.00
75%        10.00
max        21.00

Movie Interaction Distribution:
count    1000.00
mean       79.65
std         8.56
min        57.00
25%        74.00
50%        80.00
75%        85.00
max       107.00

Top 10 Most-Interacted Movies:
  movie_id   movie_title  interaction_count
movie_0074      Dark Ice                107
movie_0695 Storm Journey                105
movie_0939    Dragon War                103
movie_0398    Hero Story                102
movie_0716   Love Battle                102
movie_0285 Dragon Dragon                101
movie_0590   Secret City      

In [4]:
# 4. POPULARITY-BASED RECOMMENDATION BASELINE
# ----------------------------------------------------

# Aggregating movie-level engagement from training set
popularity_df = (
    train_df.groupby("movie_id")
    .agg(unique_users=("user_id", "nunique"),interactions=("user_id", "size"),avg_iei=("iei_score", "mean"))
    .reset_index())

# Filtering for minimum interaction threshold 
min_interaction_threshold = 10
popularity_df = popularity_df[popularity_df["interactions"] >= min_interaction_threshold].copy()

# Computing Hybrid Popularity Score (60% Engagement Quality + 40% Reach Volume)
max_unique = popularity_df["unique_users"].max()
if max_unique > 0:
    popularity_df["normalized_reach"] = popularity_df["unique_users"] / max_unique
else:
    popularity_df["normalized_reach"] = 0.0

popularity_df["popularity_score"] = (
    0.60 * popularity_df["avg_iei"] + 0.40 * popularity_df["normalized_reach"])

# Ranking Movies deterministically
popularity_df["rank"] = (
    popularity_df["popularity_score"].rank(method="first", ascending=False).astype(int))
popularity_df = popularity_df.sort_values("rank").reset_index(drop=True)

# Merging Metadata for Display & Evaluation
popularity_df = popularity_df.merge(
    movies_clean[["movie_id", "title", "genre_primary", "imdb_rating"]],on="movie_id",how="left")

# Exporting Baseline Artifacts to PROCESSED_DIR & MODELS_DIR
popularity_df.to_csv(PROCESSED_DIR / "popularity_baseline.csv", index=False)

popularity_model_artifact = {
    "popularity_df": popularity_df,
    "top_movie_ids": popularity_df["movie_id"].tolist(),
    "threshold": min_interaction_threshold,
    "weights": {"avg_iei": 0.60, "normalized_reach": 0.40}
}
joblib.dump(popularity_model_artifact, MODELS_DIR / "popularity_model.joblib")


# Production Popularity Baseline Recommendation Function
def recommend_popular(top_n: int = 10, exclude_movie_ids: list = None) -> pd.DataFrame:
    df_filtered = popularity_df.copy()
    if exclude_movie_ids:
        df_filtered = df_filtered[~df_filtered["movie_id"].isin(exclude_movie_ids)]
    
    return (
        df_filtered
        .head(top_n)[["rank", "movie_id", "title", "genre_primary", "imdb_rating", "popularity_score"]]
        .reset_index(drop=True)
    )


# Output Summary & Leaderboard Inspection
print("POPULARITY BASELINE MODEL SUMMARY")
print(f"Eligible Movies (Interactions >= {min_interaction_threshold}) : {len(popularity_df):,}")
print("\nTOP 10 POPULAR MOVIES LEADERBOARD")
print(
    popularity_df[[
        "rank", "movie_id", "title", "genre_primary",
        "unique_users", "avg_iei", "popularity_score"
    ]].head(10).to_string(index=False))

print("\nExample Baseline Recommendations (Top 5):")
print(recommend_popular(top_n=5).to_string(index=False))

print(f"\nBaseline artifacts successfully exported to:\n - {PROCESSED_DIR / 'popularity_baseline.csv'}\n - {MODELS_DIR / 'popularity_model.joblib'}")

POPULARITY BASELINE MODEL SUMMARY
Eligible Movies (Interactions >= 10) : 1,000

TOP 10 POPULAR MOVIES LEADERBOARD
 rank   movie_id           title genre_primary  unique_users  avg_iei  popularity_score
    1 movie_0787       An Family     Animation            91 0.799261          0.819743
    2 movie_0750      City Dream       History            93 0.783548          0.817792
    3 movie_0534      Big Family           War            92 0.784620          0.814697
    4 movie_0087       Our Storm        Comedy            95 0.763035          0.812961
    5 movie_0980      War Battle     Adventure            88 0.794539          0.805695
    6 movie_0075        War Hero     Animation            93 0.761983          0.804854
    7 movie_0564 Mystery Mystery   Documentary            85 0.806588          0.801710
    8 movie_0520       Big Queen     Biography            95 0.736888          0.797273
    9 movie_0129    Mission Fire        Sci-Fi            95 0.731927          0.794296
   10 

In [5]:
# 5. ITEM–ITEM COLLABORATIVE FILTERING
# -----------------------------------------------

# Transposing Interaction Matrix to Movie × User space
movie_user_matrix = interaction_matrix.T
print("ITEM–ITEM COLLABORATIVE FILTERING MATRIX COMPUTATION")
print(f"Movie-User Matrix Dimensions : {movie_user_matrix.shape[0]:,} Movies × {movie_user_matrix.shape[1]:,} Users")

# Calculating Dense Item-Item Cosine Similarity
item_similarity_array = cosine_similarity(movie_user_matrix.values)
item_similarity_df = pd.DataFrame(item_similarity_array,index=movie_user_matrix.index,columns=movie_user_matrix.index)
print(f"Item Similarity Matrix Shape : {item_similarity_df.shape[0]:,} × {item_similarity_df.shape[1]:,}")

# Exporting Item Similarity Matrix & Model Artifacts
similarity_artifact = {
    "similarity_df": item_similarity_df,
    "movie_ids": list(item_similarity_df.index),
    "matrix_shape": item_similarity_df.shape}
joblib.dump(similarity_artifact, MODELS_DIR / "item_similarity.joblib")
joblib.dump(item_similarity_df, MODELS_DIR / "item_item_cf_model.joblib")


# Item-to-Item Recommendation Function (Single Seed Movie)
def recommend_similar_movies(movie_id: str, top_n: int = 10, exclude_seen: bool = True) -> pd.DataFrame:
    if movie_id not in item_similarity_df.index:
        print(f"Warning: movie_id '{movie_id}' not found in training matrix. Falling back to Popularity Baseline.")
        return recommend_popular(top_n=top_n)

    scores = item_similarity_df[movie_id].drop(labels=[movie_id])
    top_movies = scores.sort_values(ascending=False).head(top_n).reset_index()
    top_movies.columns = ["movie_id", "similarity_score"]

    return top_movies.merge(
        movies_clean[["movie_id", "title", "genre_primary", "imdb_rating"]],on="movie_id",how="left")


# Personalized User-Level Item-CF Recommendation Function
def recommend_for_user_item_cf(user_id: str, top_n: int = 10) -> pd.DataFrame:
    if user_id not in interaction_matrix.index:
        return recommend_popular(top_n=top_n)

    # Getting user's non-zero watched items and IEI weights
    user_profile = interaction_matrix.loc[user_id]
    watched_items = user_profile[user_profile > 0]

    if watched_items.empty:
        return recommend_popular(top_n=top_n)

    # Aggregate weighted item similarities
    weighted_sims = item_similarity_df[watched_items.index].dot(watched_items)
    
    # Exclude already watched movies
    candidate_scores = weighted_sims.drop(labels=watched_items.index, errors="ignore")
    top_candidates = candidate_scores.sort_values(ascending=False).head(top_n).reset_index()
    top_candidates.columns = ["movie_id", "cf_score"]

    return top_candidates.merge(
        movies_clean[["movie_id", "title", "genre_primary", "imdb_rating"]],on="movie_id",how="left")

# Test & Validation
sample_movie_id = popularity_df.iloc[0]["movie_id"]
sample_movie_title = movies_clean.loc[movies_clean["movie_id"] == sample_movie_id, "title"].values[0]

print(f"\nSeed Movie Selected for Test : {sample_movie_title} (ID: {sample_movie_id})")
print("\nTOP 10 SIMILAR MOVIES (ITEM-ITEM CF)")
print(recommend_similar_movies(sample_movie_id, top_n=10).to_string(index=False))

print(f"\nModel artifacts successfully serialized to:\n - {MODELS_DIR / 'item_similarity.joblib'}\n - {MODELS_DIR / 'item_item_cf_model.joblib'}")

ITEM–ITEM COLLABORATIVE FILTERING MATRIX COMPUTATION
Movie-User Matrix Dimensions : 1,000 Movies × 9,997 Users
Item Similarity Matrix Shape : 1,000 × 1,000

Seed Movie Selected for Test : An Family (ID: movie_0787)

TOP 10 SIMILAR MOVIES (ITEM-ITEM CF)
  movie_id  similarity_score           title genre_primary  imdb_rating
movie_0063          0.054172    Hero Kingdom     Adventure          6.9
movie_0560          0.051404       Ice Queen     Adventure          NaN
movie_0238          0.049194  Bright Journey     Adventure          8.7
movie_0839          0.047455     Journey War           War          8.4
movie_0766          0.044280  Mission Empire       Fantasy          8.4
movie_0913          0.041317   Empire Secret         Crime          7.8
movie_0312          0.039210     King Legend       Romance          7.4
movie_0333          0.038902       The Dream        Action          8.0
movie_0111          0.038451 Kingdom Mission         Sport          NaN
movie_0784          0.03844

In [6]:
# 6. CONTENT-BASED RECOMMENDATION ENGINE
# ----------------------------------------------

# Combining Metadata Features into Composite Text Representations
metadata_cols = ["genre_primary", "genre_secondary", "language", "country_of_origin", "content_type"]
for col in metadata_cols:
    if col not in movies_clean.columns:
        movies_clean[col] = ""

movies_clean["combined_features"] = (
    movies_clean["genre_primary"].fillna("") + " " +
    movies_clean["genre_secondary"].fillna("") + " " +
    movies_clean["language"].fillna("") + " " +
    movies_clean["country_of_origin"].fillna("") + " " +
    movies_clean["content_type"].fillna("")).str.strip().str.replace(r"\s+", " ", regex=True)

# Fitting TF-IDF Vectorizer on Combined Metadata Corpus
tfidf = TfidfVectorizer(stop_words="english", min_df=1)
tfidf_matrix = tfidf.fit_transform(movies_clean["combined_features"])

# Computing Metadata Pairwise Cosine Similarity
content_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
content_sim_df = pd.DataFrame(content_sim_matrix,index=movies_clean["movie_id"],columns=movies_clean["movie_id"])

# Serializing TF-IDF Vectorizer, Matrices, and Content Models
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.joblib")
sp.save_npz(MODELS_DIR / "tfidf_matrix.npz", tfidf_matrix)

content_artifact = {
    "content_sim_df": content_sim_df,
    "feature_names": tfidf.get_feature_names_out(),
    "movie_ids": list(content_sim_df.index)
}
joblib.dump(content_artifact, MODELS_DIR / "content_similarity.joblib")
joblib.dump(content_sim_df, MODELS_DIR / "content_based_model.joblib")


# Content-Based Item Similarity Function (Seed Movie)
def recommend_content_similar(movie_id: str, top_n: int = 10) -> pd.DataFrame:
    if movie_id not in content_sim_df.index:
        print(f"Warning: movie_id '{movie_id}' not found in metadata index. Falling back to Popularity Baseline.")
        return recommend_popular(top_n=top_n)

    scores = content_sim_df[movie_id].drop(labels=[movie_id])
    top_movies = scores.sort_values(ascending=False).head(top_n).reset_index()
    top_movies.columns = ["movie_id", "content_similarity"]

    return top_movies.merge(
        movies_clean[["movie_id", "title", "genre_primary", "genre_secondary", "imdb_rating"]],on="movie_id",how="left")


# Content-Based User Profile Recommendation Function
def recommend_for_user_content_cb(user_id: str, top_n: int = 10) -> pd.DataFrame:
    if user_id not in interaction_matrix.index:
        return recommend_popular(top_n=top_n)

    user_profile = interaction_matrix.loc[user_id]
    watched_items = user_profile[user_profile > 0]

    if watched_items.empty:
        return recommend_popular(top_n=top_n)

    # Compute weighted content similarity across user's history
    valid_watched = watched_items.index.intersection(content_sim_df.index)
    if valid_watched.empty:
        return recommend_popular(top_n=top_n)

    weighted_scores = content_sim_df[valid_watched].dot(watched_items[valid_watched])
    candidate_scores = weighted_scores.drop(labels=valid_watched, errors="ignore")

    top_candidates = candidate_scores.sort_values(ascending=False).head(top_n).reset_index()
    top_candidates.columns = ["movie_id", "cb_score"]

    return top_candidates.merge(
        movies_clean[["movie_id", "title", "genre_primary", "genre_secondary", "imdb_rating"]],on="movie_id",how="left")


# Audit Output & Demonstration
print("CONTENT-BASED ENGINE SUMMARY & TF-IDF STATS")
print(f"Cleaned Movies Indexed        : {len(movies_clean):,}")
print(f"TF-IDF Feature Matrix Shape   : {tfidf_matrix.shape[0]:,} Rows × {tfidf_matrix.shape[1]:,} Features")
print(f"TF-IDF Vocabulary Size        : {len(tfidf.vocabulary_):,} Unique Terms")
print(f"Content Similarity Matrix     : {content_sim_df.shape[0]:,} × {content_sim_df.shape[1]:,}")

# Test Recommendation Preview
sample_movie_id = popularity_df.iloc[0]["movie_id"]
sample_movie_title = movies_clean.loc[movies_clean["movie_id"] == sample_movie_id, "title"].values[0]

print(f"\nSeed Movie Selected for Test  : {sample_movie_title} (ID: {sample_movie_id})")
print("\nTOP 10 METADATA SIMILAR MOVIES (CONTENT-BASED)")
print(recommend_content_similar(sample_movie_id, top_n=10).to_string(index=False))

print(f"\nContent model artifacts successfully serialized to:\n - {MODELS_DIR / 'tfidf_vectorizer.joblib'}\n - {MODELS_DIR / 'content_similarity.joblib'}\n - {MODELS_DIR / 'content_based_model.joblib'}")

CONTENT-BASED ENGINE SUMMARY & TF-IDF STATS
Cleaned Movies Indexed        : 1,000
TF-IDF Feature Matrix Shape   : 1,000 Rows × 43 Features
TF-IDF Vocabulary Size        : 43 Unique Terms
Content Similarity Matrix     : 1,000 × 1,000

Seed Movie Selected for Test  : An Family (ID: movie_0787)

TOP 10 METADATA SIMILAR MOVIES (CONTENT-BASED)
  movie_id  content_similarity           title genre_primary genre_secondary  imdb_rating
movie_0895            0.696971       House War     Animation             NaN          7.5
movie_0869            0.696971        Old City     Animation             NaN          5.8
movie_0343            0.696971  Legend Mission     Animation             NaN          NaN
movie_0738            0.696971   Mission Storm     Animation             NaN          NaN
movie_0620            0.696971     Dark Secret     Animation             NaN          7.9
movie_0243            0.684253       New Quest        Comedy             NaN          6.9
movie_0087            0.68425

In [7]:
# 7. PERSONALIZED USER TASTE PROFILES
# -----------------------------------------

from sklearn.preprocessing import normalize

# Building User-Item Interaction Matrix from Training Data
interaction_pivot = train_df.pivot_table(index="user_id",columns="movie_id",values="iei_score",aggfunc="mean",fill_value=0)

# Aligning Movies between Interaction Matrix and TF-IDF Corpus
common_movies = interaction_pivot.columns.intersection(movies_clean["movie_id"])
interaction_aligned = interaction_pivot[common_movies]

# Mapping movie IDs to their respective row index in tfidf_matrix
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movies_clean["movie_id"])}
tfidf_indices = [movie_to_idx[movie_id] for movie_id in common_movies]
tfidf_aligned = tfidf_matrix[tfidf_indices]

# User Taste Profile Computation: User Weights × Movie Feature Vectors
user_profile_raw = sp.csr_matrix(interaction_aligned.values) @ tfidf_aligned

# L2 Normalization of User Profile Vectors
user_profile_dense = user_profile_raw.toarray() if sp.issparse(user_profile_raw) else user_profile_raw
user_profile_normalized = normalize(user_profile_dense, norm="l2", axis=1)

# Converting to DataFrame and Indexing
user_profiles_df = pd.DataFrame(user_profile_normalized,index=interaction_pivot.index,columns=tfidf.get_feature_names_out())

# Serializing User Profiles and Model Artifacts
user_profile_artifact = {
    "user_profiles_df": user_profiles_df,
    "user_ids": list(user_profiles_df.index),
    "feature_names": tfidf.get_feature_names_out(),
    "common_movies": list(common_movies)
}
joblib.dump(user_profile_artifact, MODELS_DIR / "user_profiles.joblib")
user_profiles_df.to_csv(PROCESSED_DIR / "user_profiles.csv")


# Profile-Based Real-time Recommendation Function
def recommend_from_user_profile(user_id: str, top_n: int = 10) -> pd.DataFrame:
    if user_id not in user_profiles_df.index:
        print(f"Warning: User '{user_id}' not found in training profiles. Falling back to Popularity Baseline.")
        return recommend_popular(top_n=top_n)

    # User profile vector (1 x F)
    u_vector = user_profiles_df.loc[[user_id]].values
    
    # Cosine score against all candidate movies in tfidf_matrix
    scores = cosine_similarity(u_vector, tfidf_matrix).flatten()
    
    # Exclude already watched movies
    watched_movie_ids = set(train_df[train_df["user_id"] == user_id]["movie_id"])
    movie_scores = pd.DataFrame({"movie_id": movies_clean["movie_id"],"profile_match_score": scores})
    
    # Filter out watched
    candidate_movies = movie_scores[~movie_scores["movie_id"].isin(watched_movie_ids)]
    top_candidates = candidate_movies.sort_values("profile_match_score", ascending=False).head(top_n)

    return top_candidates.merge(
        movies_clean[["movie_id", "title", "genre_primary", "genre_secondary", "imdb_rating"]],on="movie_id",how="left")


# Output Audit & Test Preview
print("USER TASTE PROFILES SUMMARY")
print(f"Training Users Profiled       : {len(user_profiles_df):,}")
print(f"Common Aligned Movies         : {len(common_movies):,}")
print(f"TF-IDF Feature Dimensions     : {tfidf_aligned.shape[1]:,}")
print(f"User Taste Matrix Shape       : {user_profiles_df.shape[0]:,} Users × {user_profiles_df.shape[1]:,} Features")

# Test Recommendation for Most Active Training User
sample_user_id = train_df["user_id"].value_counts().index[0]
print(f"\nSample Active User Selected   : {sample_user_id}")
print("\nTOP 10 PROFILE-MATCHED RECOMMENDATIONS FOR USER")
print(recommend_from_user_profile(sample_user_id, top_n=10).to_string(index=False))

print(f"\nUser taste profiles successfully exported to:\n - {MODELS_DIR / 'user_profiles.joblib'}\n - {PROCESSED_DIR / 'user_profiles.csv'}")

USER TASTE PROFILES SUMMARY
Training Users Profiled       : 9,997
Common Aligned Movies         : 1,000
TF-IDF Feature Dimensions     : 43
User Taste Matrix Shape       : 9,997 Users × 43 Features

Sample Active User Selected   : user_05600

TOP 10 PROFILE-MATCHED RECOMMENDATIONS FOR USER
  movie_id  profile_match_score          title genre_primary genre_secondary  imdb_rating
movie_0512             0.616789    An Princess       Romance             NaN          0.7
movie_0583             0.614682       Hero Ice        Comedy             NaN          6.8
movie_0163             0.614682     Bright War        Comedy             NaN          5.7
movie_0188             0.614682       Our Hero        Comedy             NaN          9.3
movie_0885             0.614682    Battle Fire        Comedy             NaN          NaN
movie_0744             0.610227    A Adventure   Documentary             NaN          NaN
movie_0860             0.607145       Our Love     Animation             NaN    

In [8]:
# 8. HYBRID RECOMMENDATION ENGINE
# ----------------------------------------------

def recommend_hybrid(
    user_id: str = None,
    seed_movie_id: str = None,
    top_n: int = 10,
    w_collab: float = 0.45,
    w_content: float = 0.35,
    w_pop: float = 0.20
) -> pd.DataFrame:
    all_movies = movies_clean["movie_id"].values

    # Base Popularity Scores across all candidate movies
    pop_scores = (
        popularity_df.set_index("movie_id")["popularity_score"].reindex(all_movies, fill_value=0.0))

    # Cold-Start Check
    is_cold_user = (user_id is None) or (user_id not in interaction_matrix.index)

    # Fallback to Popularity if Cold-Start User and no Seed Movie is given
    if is_cold_user and seed_movie_id is None:
        result = (
            pop_scores.sort_values(ascending=False).head(top_n).rename("hybrid_score")
            .reset_index().rename(columns={"index": "movie_id"}))
        return result.merge(
            movies_clean[["movie_id", "title", "genre_primary", "imdb_rating"]],on="movie_id",how="left")

    # Initialize Component Scores
    collab_scores = pd.Series(0.0, index=all_movies)
    content_scores = pd.Series(0.0, index=all_movies)

    # Mode A: Seed Movie Contextual Recommendation
    if seed_movie_id is not None:
        if seed_movie_id in item_similarity_df.index:
            collab_scores = item_similarity_df.loc[seed_movie_id].reindex(all_movies, fill_value=0.0)
        if seed_movie_id in content_sim_df.index:
            content_scores = content_sim_df.loc[seed_movie_id].reindex(all_movies, fill_value=0.0)

    # Mode B: Personalized User Profile Recommendation
    elif not is_cold_user:
        user_interactions = interaction_matrix.loc[user_id]
        watched_items = user_interactions[user_interactions > 0].index

        # Collaborative Filtering Score Vector
        valid_items = watched_items.intersection(item_similarity_df.index)
        if len(valid_items) > 0:
            weights = user_interactions[valid_items]
            similarity_block = item_similarity_df.loc[valid_items].reindex(columns=all_movies, fill_value=0.0)
            collab_scores = (similarity_block.T @ weights) / (weights.sum() + 1e-8)

        # Content-Based TF-IDF Profile Matching
        if "user_profiles_df" in globals() and user_id in user_profiles_df.index:
            user_vector = user_profiles_df.loc[user_id].values.reshape(1, -1)
            content_similarity = cosine_similarity(user_vector, tfidf_matrix).flatten()
            content_scores = pd.Series(
                content_similarity, index=movies_clean["movie_id"]).reindex(all_movies, fill_value=0.0)

    # Scoring Normalization Helper
    def normalize_max(series: pd.Series) -> pd.Series:
        max_val = series.max()
        if max_val > 0:
            return series / max_val
        return series

    collab_scores = normalize_max(collab_scores)
    content_scores = normalize_max(content_scores)
    pop_scores = normalize_max(pop_scores)

    # Hybrid Weighted Score Computation
    hybrid_score = (w_collab * collab_scores +w_content * content_scores +w_pop * pop_scores)

    # Excluding Already Watched Items
    if not is_cold_user:
        user_watched = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index
        hybrid_score = hybrid_score.drop(labels=user_watched, errors="ignore")

    # Excluding Seed Movie itself if present
    if seed_movie_id is not None:
        hybrid_score = hybrid_score.drop(labels=[seed_movie_id], errors="ignore")

    # Constructing Final Recommendation Output Table
    result = pd.DataFrame({
        "movie_id": hybrid_score.index.to_numpy(),
        "collaborative_score": collab_scores.reindex(hybrid_score.index).round(4).to_numpy(),
        "content_score": content_scores.reindex(hybrid_score.index).round(4).to_numpy(),
        "popularity_score": pop_scores.reindex(hybrid_score.index).round(4).to_numpy(),
        "hybrid_score": hybrid_score.round(4).to_numpy()
    }).reset_index(drop=True)

    # Merging Movie Metadata
    result = result.merge(
        movies_clean[["movie_id", "title", "genre_primary", "imdb_rating"]],on="movie_id",how="left",validate="many_to_one")

    # Rank and Return Top N
    result = result.sort_values("hybrid_score", ascending=False).head(top_n).reset_index(drop=True)
    return result[[
        "movie_id", "title", "genre_primary", "imdb_rating",
        "collaborative_score", "content_score", "popularity_score", "hybrid_score"]]


# Export Model Configuration Artifact
hybrid_config = {
    "weights": {"w_collab": 0.45, "w_content": 0.35, "w_pop": 0.20},
    "default_top_n": 10,
    "model_components": ["Item_CF", "TFIDF_Content", "Popularity_Baseline"]
}
joblib.dump(hybrid_config, MODELS_DIR / "hybrid_model_config.joblib")

# Execution & Multi-Scenario Test Audit
print("HYBRID RECOMMENDATION ENGINE TEST AUDIT")

# Test 1: Personalized Existing User Mode
sample_user = train_df["user_id"].value_counts().index[0]
print(f"\n1. PERSONALIZED HYBRID RECOMMENDATIONS (User ID: {sample_user}):")
print(recommend_hybrid(user_id=sample_user, top_n=5).to_string(index=False))

# Test 2: Seed-Movie Contextual Recommendation Mode
sample_movie = popularity_df.iloc[0]["movie_id"]
sample_title = movies_clean.loc[movies_clean["movie_id"] == sample_movie, "title"].values[0]
print(f"\n2. SEED-MOVIE HYBRID RECOMMENDATIONS (Seed: '{sample_title}', ID: {sample_movie}):")
print(recommend_hybrid(seed_movie_id=sample_movie, top_n=5).to_string(index=False))

# Test 3: Cold-Start Fallback Mode
print("\n3. COLD-START USER FALLBACK HYBRID RECOMMENDATIONS:")
print(recommend_hybrid(user_id="UNKNOWN_USER_999", top_n=5).to_string(index=False))

print(f"\nHybrid model configuration saved to: {MODELS_DIR / 'hybrid_model_config.joblib'}")

HYBRID RECOMMENDATION ENGINE TEST AUDIT

1. PERSONALIZED HYBRID RECOMMENDATIONS (User ID: user_05600):
  movie_id            title genre_primary  imdb_rating  collaborative_score  content_score  popularity_score  hybrid_score
movie_0860         Our Love     Animation          NaN               0.0809         0.9413            0.8601        0.5379
movie_0607          The War       Romance          7.6               0.0975         0.8923            0.8561        0.5274
movie_0214      Last Dragon     Animation          4.3               0.0744         0.9140            0.8632        0.5260
movie_0559   Mystery Dragon       Romance          NaN               0.0966         0.9243            0.7885        0.5247
movie_0654 Adventure Legend         Drama          6.5               0.0676         0.8994            0.8800        0.5212

2. SEED-MOVIE HYBRID RECOMMENDATIONS (Seed: 'An Family', ID: movie_0787):
  movie_id          title genre_primary  imdb_rating  collaborative_score  content_s

In [9]:
# 9. TEST HYBRID RECOMMENDATION
# ----------------------------------

# Targeting test parameters
test_user = "user_01741"
test_seed_movie = "movie_0787"

# Fetching Seed Movie Metadata for clear reporting
seed_movie_info = movies_clean[movies_clean["movie_id"] == test_seed_movie]
seed_title = seed_movie_info["title"].values[0] if not seed_movie_info.empty else "Unknown Title"
seed_genre = seed_movie_info["genre_primary"].values[0] if not seed_movie_info.empty else "Unknown Genre"

# Generating hybrid recommendations
recommendations = recommend_hybrid(user_id=test_user,seed_movie_id=test_seed_movie,top_n=10)

# Exporting test output artifact
recommendations.to_csv(METRICS_DIR / "hybrid_test_recommendations.csv", index=False)

# Displaying Audit & User Context Summary
print("HYBRID RECOMMENDATION SYSTEM TEST & INSPECTION")
print(f"Target Test User ID  : {test_user}")
print(f"Target Seed Movie    : {seed_title} (ID: {test_seed_movie} | Genre: {seed_genre})")

# Displaying Target User's Watch History (if user exists in training set)
if test_user in interaction_matrix.index:
    user_history = (
        train_df[train_df["user_id"] == test_user]
        .merge(movies_clean[["movie_id", "title", "genre_primary"]], on="movie_id", how="left")
        .sort_values("iei_score", ascending=False).head(5))
    
    print(f"\nUser {test_user}'s Top Watched Movies (Historical Context):")
    print(user_history[["movie_id", "title", "genre_primary", "iei_score"]].to_string(index=False))
else:
    print(f"\nUser {test_user} is not present in the training set (Operating in Cold-Start mode).")

print("TOP 10 HYBRID RECOMMENDATION RESULTS")
print(recommendations.to_string(index=False))
print(f"\nTest output successfully saved to: {METRICS_DIR / 'hybrid_test_recommendations.csv'}")

HYBRID RECOMMENDATION SYSTEM TEST & INSPECTION
Target Test User ID  : user_01741
Target Seed Movie    : An Family (ID: movie_0787 | Genre: Animation)

User user_01741's Top Watched Movies (Historical Context):
  movie_id         title genre_primary  iei_score
movie_0871 Princess City     Animation   0.936125
movie_0432  Love Journey     Animation   0.850000
movie_0720    Legend War       Western   0.562125
movie_0552     A Mission       Romance   0.407159
movie_0926  Family Queen        Sci-Fi   0.251600
TOP 10 HYBRID RECOMMENDATION RESULTS
  movie_id          title genre_primary  imdb_rating  collaborative_score  content_score  popularity_score  hybrid_score
movie_0087      Our Storm        Comedy          6.6               0.0000         0.6843            0.9917        0.4378
movie_0869       Old City     Animation          5.8               0.0166         0.6970            0.8670        0.4248
movie_0343 Legend Mission     Animation          NaN               0.0197         0.6970  

In [10]:
# 10. RECOMMENDATION EVALUATION (TOP-K RANKING METRICS)
# --------------------------------------------------------

import math
import matplotlib.pyplot as plt

# Standardizing Evaluation Cutoff & Test User Cohort
K = 10
test_users = test_df["user_id"].unique()

# Sample evaluation users for deterministic, efficient computation
np.random.seed(42)
eval_users = np.random.choice(test_users, size=min(300, len(test_users)), replace=False)

user_precisions = []
user_recalls = []
user_aps = []
user_ndcgs = []
user_hits = []
user_eval_records = []

# Iterating User Evaluation Loop
for user_id in eval_users:
    actual_items = set(test_df.loc[test_df["user_id"] == user_id, "movie_id"])
    if not actual_items:
        continue

    # Generating Top-K recommendations using trained hybrid engine
    recs = recommend_hybrid(user_id=user_id, top_n=K)
    
    if recs.empty or "movie_id" not in recs.columns:
        continue

    recommended_list = recs["movie_id"].tolist()

    # Binary hits array
    hits_list = [1 if item in actual_items else 0 for item in recommended_list]
    num_hits = sum(hits_list)
    hit_flag = 1 if num_hits > 0 else 0

    # Per-user Precision & Recall
    precision = num_hits / K
    recall = num_hits / len(actual_items)

    # Average Precision@K (AP@K)
    if num_hits > 0:
        running_hits = 0
        ap_score = 0.0
        for rank, hit in enumerate(hits_list, start=1):
            if hit:
                running_hits += 1
                ap_score += running_hits / rank
        ap_at_k = ap_score / min(len(actual_items), K)
    else:
        ap_at_k = 0.0

    # Discounted Cumulative Gain@K (DCG@K & NDCG@K)
    dcg = sum([hit / math.log2(rank + 1) for rank, hit in enumerate(hits_list, start=1)])
    idcg = sum([1.0 / math.log2(rank + 1) for rank in range(1, min(len(actual_items), K) + 1)])
    ndcg = dcg / idcg if idcg > 0 else 0.0

    # Append to macro accumulators
    user_precisions.append(precision)
    user_recalls.append(recall)
    user_aps.append(ap_at_k)
    user_ndcgs.append(ndcg)
    user_hits.append(hit_flag)

    user_eval_records.append({
        "user_id": user_id,
        "num_actual_items": len(actual_items),
        "hits_at_k": num_hits,
        "precision_at_k": precision,
        "recall_at_k": recall,
        "ap_at_k": ap_at_k,
        "ndcg_at_k": ndcg
    })

# Computing Macro-Averaged Evaluation Scores
macro_precision = float(np.mean(user_precisions)) if user_precisions else 0.0
macro_recall = float(np.mean(user_recalls)) if user_recalls else 0.0
map_at_k = float(np.mean(user_aps)) if user_aps else 0.0
macro_ndcg = float(np.mean(user_ndcgs)) if user_ndcgs else 0.0
hit_rate_at_k = float(np.mean(user_hits)) if user_hits else 0.0

# Exporting Metric Artifacts to METRICS_DIR
evaluation_metrics_summary = {
    "model_evaluated": "Hybrid_Recommendation_Engine",
    "evaluated_users_count": len(user_precisions),
    "cutoff_k": K,
    "precision_at_k": round(macro_precision, 4),
    "recall_at_k": round(macro_recall, 4),
    "map_at_k": round(map_at_k, 4),
    "ndcg_at_k": round(macro_ndcg, 4),
    "hit_rate_at_k": round(hit_rate_at_k, 4)
}

with open(METRICS_DIR / "hybrid_evaluation_metrics.json", "w") as f:
    json.dump(evaluation_metrics_summary, f, indent=4)

user_scores_df = pd.DataFrame(user_eval_records)
user_scores_df.to_csv(METRICS_DIR / "hybrid_user_evaluation_scores.csv", index=False)

# Plotting Evaluation Summary Chart
plt.figure(figsize=(9, 5))
metrics_names = [f"Precision@{K}", f"Recall@{K}", f"MAP@{K}", f"NDCG@{K}", f"Hit Rate@{K}"]
metrics_values = [macro_precision, macro_recall, map_at_k, macro_ndcg, hit_rate_at_k]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

bars = plt.bar(metrics_names, metrics_values, color=colors, width=0.55, edgecolor="black", alpha=0.85)
plt.title(f"Hybrid Recommendation Engine Evaluation (K={K})", fontsize=13, fontweight="bold", pad=12)
plt.ylabel("Score", fontsize=11)
plt.ylim(0, max(metrics_values) * 1.25 if max(metrics_values) > 0 else 1.0)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f"{yval:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

save_fig("hybrid_recommendation_evaluation_metrics")
plt.close()

# Audit & Results Output
print(f"Evaluated Test Users Cohort      : {len(user_precisions):,}")
print(f"Target Ranking Cutoff (K)        : {K}")
print(f"Precision@{K:<2}                 : {macro_precision:.4f}")
print(f"Recall@{K:<2}                    : {macro_recall:.4f}")
print(f"MAP@{K:<2}                       : {map_at_k:.4f}")
print(f"NDCG@{K:<2}                      : {macro_ndcg:.4f}")
print(f"Hit Rate@{K:<2}                  : {hit_rate_at_k:.4f}")
print(f"Metrics JSON exported to: {METRICS_DIR / 'hybrid_evaluation_metrics.json'}")
print(f"User log CSV exported to: {METRICS_DIR / 'hybrid_user_evaluation_scores.csv'}")
print(f"Diagnostic chart saved to: {FIGURES_DIR / 'hybrid_recommendation_evaluation_metrics.png'}")

Figure saved: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/figures/hybrid_recommendation_evaluation_metrics.png
Evaluated Test Users Cohort      : 300
Target Ranking Cutoff (K)        : 10
Precision@10                 : 0.0023
Recall@10                    : 0.0111
MAP@10                       : 0.0045
NDCG@10                      : 0.0068
Hit Rate@10                  : 0.0233
Metrics JSON exported to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/metrics/hybrid_evaluation_metrics.json
User log CSV exported to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/metrics/hybrid_user_evaluation_scores.csv
Diagnostic chart saved to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/figures/hybrid_recommendation_evaluation_metrics.png


In [11]:
# 11. HYBRID WEIGHT OPTIMIZATION (GRID SEARCH)
# -------------------------------------------------

import time
import matplotlib.pyplot as plt

start_time = time.time()

# Generating valid weight triplets summing to 1.0 (0.1 step size)
weight_candidates = []
for c in range(1, 9):
    for t in range(1, 9):
        p = 10 - c - t
        if p >= 1:
            weight_candidates.append((round(c * 0.1, 2), round(t * 0.1, 2), round(p * 0.1, 2)))

grid_results = []
total_candidates = len(weight_candidates)

print(f"Starting Grid Search across {total_candidates} candidate weight configurations...")

# Grid Search Iteration Loop
for idx, (wc, wt, wp) in enumerate(weight_candidates, start=1):
    user_aps = []
    
    for user_id in eval_users:
        actual_items = set(test_df.loc[test_df["user_id"] == user_id, "movie_id"])
        if not actual_items:
            continue

        recs = recommend_hybrid(user_id=user_id,top_n=K, w_collab=wc,w_content=wt,w_pop=wp)
        
        if recs.empty or "movie_id" not in recs.columns:
            user_aps.append(0.0)
            continue

        recommended_list = recs["movie_id"].tolist()
        hits_list = [1 if item in actual_items else 0 for item in recommended_list]
        num_hits = sum(hits_list)

        if num_hits > 0:
            running_hits = 0
            ap_score = 0.0
            for rank, hit in enumerate(hits_list, start=1):
                if hit:
                    running_hits += 1
                    ap_score += running_hits / rank
            user_aps.append(ap_score / min(len(actual_items), K))
        else:
            user_aps.append(0.0)

    mean_map = float(np.mean(user_aps)) if user_aps else 0.0
    grid_results.append({"w_collab": wc, "w_content": wt, "w_pop": wp, f"MAP@{K}": round(mean_map, 4)})

# Process & Rank Optimization Results
grid_df = (
    pd.DataFrame(grid_results).sort_values(by=f"MAP@{K}", ascending=False).reset_index(drop=True))

best_row = grid_df.iloc[0]
best_weights = {
    "w_collab": float(best_row["w_collab"]),
    "w_content": float(best_row["w_content"]),
    "w_pop": float(best_row["w_pop"]),
    f"optimal_map_at_{K}": float(best_row[f"MAP@{K}"])
}

# Export Artifacts to METRICS_DIR
grid_df.to_csv(METRICS_DIR / "hybrid_weight_grid_search_results.csv", index=False)

with open(METRICS_DIR / "optimal_hybrid_weights.json", "w") as f:
    json.dump(best_weights, f, indent=4)

# Plot Top 10 Weight Configurations
top10_grid = grid_df.head(10).copy()
top10_grid["config_label"] = (
    top10_grid["w_collab"].astype(str) + " / " + 
    top10_grid["w_content"].astype(str) + " / " + 
    top10_grid["w_pop"].astype(str)
)

plt.figure(figsize=(10, 5))
bars = plt.barh(
    top10_grid["config_label"][::-1],
    top10_grid[f"MAP@{K}"][::-1],
    color="#2ca02c",
    edgecolor="black",
    alpha=0.85)
plt.title(f"Top 10 Hybrid Weight Configurations by MAP@{K}", fontsize=13, fontweight="bold", pad=12)
plt.xlabel(f"MAP@{K} Score", fontsize=11)
plt.ylabel("Weights (Collab / Content / Pop)", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.5)

for bar in bars:
    xval = bar.get_width()
    plt.text(xval + 0.0005, bar.get_y() + bar.get_height()/2.0, f"{xval:.4f}", ha="left", va="center", fontsize=9, fontweight="bold")

save_fig("hybrid_weight_optimization_top10")
plt.close()

elapsed_time = time.time() - start_time

# Audit & Results Output
print("HYBRID WEIGHT OPTIMIZATION SUMMARY (GRID SEARCH)")
print(f"Total Configurations Evaluated     : {total_candidates}")
print(f"Optimization Execution Time        : {elapsed_time:.2f} seconds")
print("TOP 5 ENSEMBLE WEIGHT CONFIGURATIONS:")
print(grid_df.head(5).to_string(index=False))
print(f"Optimal Weights (Collab / Content / Pop)    : ({best_weights['w_collab']}, {best_weights['w_content']}, {best_weights['w_pop']})")
print(f"Optimized MAP@{K:<2}                        : {best_weights[f'optimal_map_at_{K}']:.4f}")
print(f"Grid search CSV exported to : {METRICS_DIR / 'hybrid_weight_grid_search_results.csv'}")
print(f"Optimal weights JSON saved to: {METRICS_DIR / 'optimal_hybrid_weights.json'}")
print(f"Diagnostic chart saved to   : {FIGURES_DIR / 'hybrid_weight_optimization_top10.png'}")

Starting Grid Search across 36 candidate weight configurations...
Figure saved: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/figures/hybrid_weight_optimization_top10.png
HYBRID WEIGHT OPTIMIZATION SUMMARY (GRID SEARCH)
Total Configurations Evaluated     : 36
Optimization Execution Time        : 136.64 seconds
TOP 5 ENSEMBLE WEIGHT CONFIGURATIONS:
 w_collab  w_content  w_pop  MAP@10
      0.7        0.1    0.2  0.0050
      0.7        0.2    0.1  0.0050
      0.8        0.1    0.1  0.0050
      0.6        0.1    0.3  0.0050
      0.2        0.6    0.2  0.0048
Optimal Weights (Collab / Content / Pop)    : (0.7, 0.1, 0.2)
Optimized MAP@10                        : 0.0050
Grid search CSV exported to : /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/metrics/hybrid_weight_grid_search_results.csv
Optimal weights JSON saved to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/metrics/optimal_hybrid_we

In [12]:
# 12. PRODUCTION ENGINE ENCAPSULATION
# -------------------------------------------------------

class StreamIntelRecommenderEngine:
    def __init__(
        self,
        interaction_matrix: pd.DataFrame,
        item_sim_df: pd.DataFrame,
        content_sim_df: pd.DataFrame,
        user_profiles: pd.DataFrame,
        popularity_df: pd.DataFrame,
        movies_df: pd.DataFrame,
        tfidf_matrix,
        weights=(0.50, 0.30, 0.20)
    ):
        self.interaction_matrix = interaction_matrix
        self.item_sim_df = item_sim_df
        self.content_sim_df = content_sim_df
        self.user_profiles = user_profiles
        self.tfidf_matrix = tfidf_matrix
        
        # Ensure movies_df retains movie_id column while enabling quick lookup
        self.movies_raw = movies_df.copy()
        if "movie_id" in movies_df.columns:
            self.movies_lookup = movies_df.set_index("movie_id")
        else:
            self.movies_lookup = movies_df

        # Popularity lookup vector
        if "popularity_score" in popularity_df.columns:
            self.popularity_series = popularity_df.set_index("movie_id")["popularity_score"]
        else:
            self.popularity_series = popularity_df

        self.all_movies = self.movies_raw["movie_id"].values if "movie_id" in self.movies_raw.columns else self.movies_raw.index.values

        # Parse weight configuration (supports dict or tuple/list)
        if isinstance(weights, dict):
            self.w_collab = float(weights.get("w_collab", 0.50))
            self.w_content = float(weights.get("w_content", 0.30))
            self.w_pop = float(weights.get("w_pop", 0.20))
        elif isinstance(weights, (tuple, list)) and len(weights) == 3:
            self.w_collab, self.w_content, self.w_pop = weights
        else:
            raise ValueError("Weights must be a dict with keys ('w_collab', 'w_content', 'w_pop') or a 3-element tuple.")

    def _normalize(self, series: pd.Series) -> pd.Series:
        max_val = series.max()
        if max_val > 0:
            return series / max_val
        return series

    def predict(
        self,
        user_id: str = None,
        seed_movie_id: str = None,
        top_n: int = 10
    ) -> pd.DataFrame:
        is_cold = (user_id is None) or (user_id not in self.interaction_matrix.index)

        # Cold-Start Fallback (Popularity Baseline)
        if is_cold and seed_movie_id is None:
            top_pop = (
                self.popularity_series.sort_values(ascending=False)
                .head(top_n)
                .reset_index()
                .rename(columns={"index": "movie_id", "popularity_score": "hybrid_score"})
            )
            top_pop["collaborative_score"] = 0.0
            top_pop["content_score"] = 0.0
            top_pop["popularity_score"] = top_pop["hybrid_score"]
            
            return top_pop.merge(
                self.movies_raw[["movie_id", "title", "genre_primary", "imdb_rating"]],on="movie_id",how="left"
            )[["movie_id", "title", "genre_primary", "imdb_rating", "collaborative_score", "content_score", "popularity_score", "hybrid_score"]]

        # Initialize Score Vectors
        collab_scores = pd.Series(0.0, index=self.all_movies)
        content_scores = pd.Series(0.0, index=self.all_movies)
        pop_scores = self.popularity_series.reindex(self.all_movies, fill_value=0.0)

        # Seed Movie Mode
        if seed_movie_id is not None:
            if seed_movie_id in self.item_sim_df.index:
                collab_scores = self.item_sim_df.loc[seed_movie_id].reindex(self.all_movies, fill_value=0.0)
            if seed_movie_id in self.content_sim_df.index:
                content_scores = self.content_sim_df.loc[seed_movie_id].reindex(self.all_movies, fill_value=0.0)

        # User Profile Mode
        elif not is_cold:
            user_watched = self.interaction_matrix.loc[user_id][self.interaction_matrix.loc[user_id] > 0]
            valid_items = user_watched.index.intersection(self.item_sim_df.index)

            # Collaborative Score
            if len(valid_items) > 0:
                weights = user_watched[valid_items]
                sim_block = self.item_sim_df.loc[valid_items].reindex(columns=self.all_movies, fill_value=0.0)
                collab_scores = (sim_block.T @ weights) / (weights.sum() + 1e-8)

            # Content Profile Score
            if user_id in self.user_profiles.index:
                u_vector = self.user_profiles.loc[user_id].values.reshape(1, -1)
                content_sims = cosine_similarity(u_vector, self.tfidf_matrix).flatten()
                content_scores = pd.Series(
                    content_sims, index=self.movies_raw["movie_id"]
                ).reindex(self.all_movies, fill_value=0.0)

        # Normalize Components
        norm_collab = self._normalize(collab_scores)
        norm_content = self._normalize(content_scores)
        norm_pop = self._normalize(pop_scores)

        # Ensemble Weighted Score Computation
        hybrid_scores = (
            self.w_collab * norm_collab +
            self.w_content * norm_content +
            self.w_pop * norm_pop
        )

        # Exclude Already Watched / Seed Movies
        if not is_cold:
            watched_ids = self.interaction_matrix.loc[user_id][self.interaction_matrix.loc[user_id] > 0].index
            hybrid_scores = hybrid_scores.drop(labels=watched_ids, errors="ignore")

        if seed_movie_id is not None:
            hybrid_scores = hybrid_scores.drop(labels=[seed_movie_id], errors="ignore")

        # Construct Output Dataframe
        output = pd.DataFrame({
            "movie_id": hybrid_scores.index.to_numpy(),
            "collaborative_score": norm_collab.reindex(hybrid_scores.index).round(4).to_numpy(),
            "content_score": norm_content.reindex(hybrid_scores.index).round(4).to_numpy(),
            "popularity_score": norm_pop.reindex(hybrid_scores.index).round(4).to_numpy(),
            "hybrid_score": hybrid_scores.round(4).to_numpy()}).reset_index(drop=True)

        output = output.merge(
            self.movies_raw[["movie_id", "title", "genre_primary", "imdb_rating"]],on="movie_id",how="left")

        return (
            output.sort_values("hybrid_score", ascending=False)
            .head(top_n)
            .reset_index(drop=True)[
                ["movie_id", "title", "genre_primary", "imdb_rating",
                 "collaborative_score", "content_score", "popularity_score", "hybrid_score"]
            ]
        )


# Instantiate Production Engine using Optimal Grid Search Weights (from Cell 11)
target_weights = best_weights if "best_weights" in globals() else (0.45, 0.35, 0.20)

engine = StreamIntelRecommenderEngine(
    interaction_matrix=interaction_matrix,
    item_sim_df=item_similarity_df,
    content_sim_df=content_sim_df,
    user_profiles=user_profiles_df if "user_profiles_df" in globals() else user_profiles,
    popularity_df=popularity_df,
    movies_df=movies_clean,
    tfidf_matrix=tfidf_matrix,
    weights=target_weights
)

# Serialize Production Engine Instance
joblib.dump(engine, MODELS_DIR / "production_recommender_engine.joblib")


# Audit Output & Test Preview
print("STREAMINTEL PRODUCTION RECOMMENDER ENGINE INITIALIZED")
print(f"Configured Weight Allocation  : Collab={engine.w_collab:.2f} | Content={engine.w_content:.2f} | Pop={engine.w_pop:.2f}")
print(f"Total Indexed Movies          : {len(engine.all_movies):,}")
print(f"Total Profiled Users          : {len(engine.user_profiles):,}")

sample_test_user = train_df["user_id"].value_counts().index[0]
print(f"\nSAMPLE INFERENCE PREDICTION (User ID: {sample_test_user}):")
print(engine.predict(user_id=sample_test_user, top_n=5).to_string(index=False))

print(f"\nProduction Engine serialized to: {MODELS_DIR / 'production_recommender_engine.joblib'}")

STREAMINTEL PRODUCTION RECOMMENDER ENGINE INITIALIZED
Configured Weight Allocation  : Collab=0.70 | Content=0.10 | Pop=0.20
Total Indexed Movies          : 1,000
Total Profiled Users          : 9,997

SAMPLE INFERENCE PREDICTION (User ID: user_05600):
  movie_id       title genre_primary  imdb_rating  collaborative_score  content_score  popularity_score  hybrid_score
movie_0890    War Fire         Music          NaN               0.1399         0.5525            0.9562        0.3444
movie_0750  City Dream       History          5.1               0.0944         0.7070            0.9976        0.3363
movie_0069    An Quest   Documentary          8.1               0.0989         0.7658            0.9338        0.3326
movie_0607     The War       Romance          7.6               0.0975         0.8923            0.8561        0.3287
movie_0771 First Quest        Horror          7.2               0.0855         0.7526            0.9578        0.3267

Production Engine serialized to: /kaggl

In [13]:
# 13. MODEL SERIALIZATION & ARTIFACT EXPORT
# -----------------------------------------------

# Ensure Target Directory Exists
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Extract Evaluation Benchmark Scores Safely
map_score = None
if "best_row" in globals():
    map_score = float(best_row[f"MAP@{K}"])
elif isinstance(best_weights, dict) and f"optimal_map_at_{K}" in best_weights:
    map_score = float(best_weights[f"optimal_map_at_{K}"])

# Construct Complete Production Artifact Dictionary
artifacts = {
    "engine": engine,
    "tfidf_vectorizer": tfidf,
    "optimal_weights": best_weights,
    "catalog_version": "1.0.0",
    "evaluation_metrics": {
        "k": K if "K" in globals() else 10,
        "map_at_k": round(map_score, 4) if map_score is not None else None}}

artifact_path = MODELS_DIR / "recommendation_engine_v1.joblib"

# Serialize Model Artifacts with Level-3 Compression
joblib.dump(artifacts, artifact_path, compress=3)
file_size_mb = os.path.getsize(artifact_path) / (1024 * 1024)

# Export Lightweight JSON Model Manifest
manifest = {
    "model_name": "StreamIntelRecommenderEngine",
    "catalog_version": "1.0.0",
    "artifact_file": "recommendation_engine_v1.joblib",
    "compressed_size_mb": round(file_size_mb, 2),
    "compression_level": 3,
    "optimal_weights": best_weights,
    "evaluation_metrics": artifacts["evaluation_metrics"]
}

manifest_path = MODELS_DIR / "model_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

# Perform Round-Trip Verification Load
verified_artifacts = joblib.load(artifact_path)
test_user_id = sample_test_user if "sample_test_user" in globals() else "user_01741"
verification_pred = verified_artifacts["engine"].predict(user_id=test_user_id, top_n=1)
top_rec_title = verification_pred["title"].values[0] if not verification_pred.empty else "N/A"

# Audit Output Summary
print("PRODUCTION MODEL SERIALIZATION & ARTIFACT EXPORT")
print(f"Exported Bundle Path       : {artifact_path}")
print(f"Compressed Artifact Size   : {file_size_mb:.2f} MB")
print(f"Catalog Model Version      : {artifacts['catalog_version']}")
print(f"Optimal Ensemble Weights   : {best_weights}")
print(f"Benchmark MAP Score        : {artifacts['evaluation_metrics']['map_at_k']}")
print(f"Round-Trip Integrity Check : PASSED")
print(f"Verification Prediction    : User '{test_user_id}' -> Top Rec: '{top_rec_title}'")
print(f"Manifest JSON saved to     : {manifest_path}")

PRODUCTION MODEL SERIALIZATION & ARTIFACT EXPORT
Exported Bundle Path       : /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/models/recommendation_engine_v1.joblib
Compressed Artifact Size   : 11.41 MB
Catalog Model Version      : 1.0.0
Optimal Ensemble Weights   : {'w_collab': 0.7, 'w_content': 0.1, 'w_pop': 0.2, 'optimal_map_at_10': 0.005}
Benchmark MAP Score        : 0.005
Round-Trip Integrity Check : PASSED
Verification Prediction    : User 'user_05600' -> Top Rec: 'War Fire'
Manifest JSON saved to     : /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/models/model_manifest.json


In [14]:
# 14. SVD MATRIX FACTORIZATION MODULE
# ----------------------------------------------------------

from sklearn.decomposition import TruncatedSVD

# Validate Target Directory
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Dynamic Latent Factor Dimension Configuration
n_users, n_items = interaction_matrix.shape
TARGET_COMPONENTS = 20
N_COMPONENTS = min(TARGET_COMPONENTS, min(n_users, n_items) - 1)

# Instantiate & Fit TruncatedSVD Model
svd_model = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

# User Latent Factor Matrix (U * Sigma): Shape (n_users, n_components)
user_latent_matrix = svd_model.fit_transform(interaction_matrix.values)

# Movie Latent Factor Matrix (V^T): Shape (n_components, n_movies)
item_latent_matrix = svd_model.components_

# Reconstruct Predicted Interaction Scores (U * Sigma * V^T)
svd_predicted_ratings = np.dot(user_latent_matrix, item_latent_matrix)

# Converting to DataFrame aligned with original interaction matrix indices
svd_pred_df = pd.DataFrame(
    svd_predicted_ratings,
    index=interaction_matrix.index,
    columns=interaction_matrix.columns
)

# Compute Latent Space Item Similarity Matrix
svd_item_sim = cosine_similarity(item_latent_matrix.T)
svd_item_sim_df = pd.DataFrame(
    svd_item_sim,
    index=interaction_matrix.columns,
    columns=interaction_matrix.columns
)

# Extract Variance Metrics
explained_variance_ratio = svd_model.explained_variance_ratio_
total_explained_variance = float(explained_variance_ratio.sum() * 100)

# Serialize SVD Model & Latent Factor Artifacts
svd_artifacts = {
    "svd_model": svd_model,
    "user_latent_matrix": user_latent_matrix,
    "item_latent_matrix": item_latent_matrix,
    "svd_pred_df": svd_pred_df,
    "svd_item_sim_df": svd_item_sim_df,
    "n_components": N_COMPONENTS,
    "total_explained_variance": total_explained_variance
}

svd_artifact_path = MODELS_DIR / "svd_latent_factors.joblib"
joblib.dump(svd_artifacts, svd_artifact_path, compress=3)

# 8. Execution Audit Summary
print("SVD MATRIX FACTORIZATION MODULE EXECUTED")
print(f"Interaction Matrix Dimensions  : {n_users:,} Users x {n_items:,} Movies")
print(f"Selected Latent Components     : {N_COMPONENTS}")
print(f"User Latent Factor Matrix      : {user_latent_matrix.shape}")
print(f"Movie Latent Factor Matrix     : {item_latent_matrix.shape}")
print(f"Reconstructed Matrix Shape     : {svd_pred_df.shape}")
print(f"Total Cumulative Variance      : {total_explained_variance:.2f}%")
print(f"Top 3 Latent Variance Breakdown: {explained_variance_ratio[:3] * 100}")
print(f"Latent Factor Artifacts saved to: {svd_artifact_path}")
print("SVD Model Trained & Serialized Successfully.")

SVD MATRIX FACTORIZATION MODULE EXECUTED
Interaction Matrix Dimensions  : 9,997 Users x 1,000 Movies
Selected Latent Components     : 20
User Latent Factor Matrix      : (9997, 20)
Movie Latent Factor Matrix     : (20, 1000)
Reconstructed Matrix Shape     : (9997, 1000)
Total Cumulative Variance      : 3.93%
Top 3 Latent Variance Breakdown: [0.12131229 0.21514245 0.21350406]
Latent Factor Artifacts saved to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/models/svd_latent_factors.joblib
SVD Model Trained & Serialized Successfully.


In [15]:
# 15. ENHANCED 4-WAY HYBRID ENGINE WITH SVD
# ----------------------------------------------------

# Ensure Target Directory Exists
MODELS_DIR.mkdir(parents=True, exist_ok=True)

class StreamIntelAdvancedRecommender:
    def __init__(
        self,
        interaction_matrix: pd.DataFrame,
        item_sim_df: pd.DataFrame,
        svd_pred_df: pd.DataFrame,
        user_profiles: pd.DataFrame,
        popularity_df: pd.DataFrame,
        movies_df: pd.DataFrame,
        content_sim_df: pd.DataFrame = None,
        tfidf_matrix=None,
        weights=(0.15, 0.35, 0.35, 0.15)
    ):
        self.interaction_matrix = interaction_matrix
        self.item_sim_df = item_sim_df
        self.svd_pred_df = svd_pred_df
        self.user_profiles = user_profiles
        self.content_sim_df = content_sim_df
        self.tfidf_matrix = tfidf_matrix

        # Normalize Movies DataFrame indexing
        self.movies_raw = movies_df.copy()
        if "movie_id" in movies_df.columns:
            self.movies_lookup = movies_df.set_index("movie_id")
        else:
            self.movies_lookup = movies_df

        # Popularity score vector lookup
        if "popularity_score" in popularity_df.columns:
            self.popularity_series = popularity_df.set_index("movie_id")["popularity_score"]
        else:
            self.popularity_series = popularity_df

        self.all_movies = self.movies_raw["movie_id"].values if "movie_id" in self.movies_raw.columns else self.movies_raw.index.values

        # Parse 4-way weight configurations (dict or tuple/list)
        if isinstance(weights, dict):
            self.w_cf = float(weights.get("w_cf", 0.15))
            self.w_svd = float(weights.get("w_svd", 0.35))
            self.w_content = float(weights.get("w_content", 0.35))
            self.w_pop = float(weights.get("w_pop", 0.15))
        elif isinstance(weights, (tuple, list)) and len(weights) == 4:
            self.w_cf, self.w_svd, self.w_content, self.w_pop = weights
        else:
            raise ValueError("Weights must be a dict with keys ('w_cf', 'w_svd', 'w_content', 'w_pop') or a 4-element tuple.")

    def _normalize(self, series: pd.Series) -> pd.Series:
        max_val = series.max()
        if max_val > 0:
            return series / max_val
        return series

    def predict(self,user_id: str = None,seed_movie_id: str = None,top_n: int = 10
    ) -> pd.DataFrame:
        is_cold = (user_id is None) or (user_id not in self.interaction_matrix.index)

        # Cold-Start Fallback (Popularity Baseline)
        if is_cold and seed_movie_id is None:
            top_pop = (
                self.popularity_series.sort_values(ascending=False)
                .head(top_n)
                .reset_index()
                .rename(columns={"index": "movie_id", "popularity_score": "hybrid_score"}))
            top_pop["cf_score"] = 0.0
            top_pop["svd_score"] = 0.0
            top_pop["content_score"] = 0.0
            top_pop["popularity_score"] = top_pop["hybrid_score"]

            cols_to_merge = ["movie_id", "title", "genre_primary"]
            if "imdb_rating" in self.movies_raw.columns:
                cols_to_merge.append("imdb_rating")

            return top_pop.merge(self.movies_raw[cols_to_merge], on="movie_id", how="left")

        # Initialize Score Components
        cf_scores = pd.Series(0.0, index=self.all_movies)
        svd_scores = pd.Series(0.0, index=self.all_movies)
        content_scores = pd.Series(0.0, index=self.all_movies)
        pop_scores = self.popularity_series.reindex(self.all_movies, fill_value=0.0)

        # SVD Latent Factor Component
        if not is_cold and user_id in self.svd_pred_df.index:
            svd_scores = self.svd_pred_df.loc[user_id].reindex(self.all_movies, fill_value=0.0)

        # Item-Item Collaborative Filtering Component
        if seed_movie_id and seed_movie_id in self.item_sim_df.index:
            cf_scores = self.item_sim_df.loc[seed_movie_id].reindex(self.all_movies, fill_value=0.0)
        elif not is_cold:
            watched = self.interaction_matrix.loc[user_id][self.interaction_matrix.loc[user_id] > 0]
            valid = watched.index.intersection(self.item_sim_df.index)
            if len(valid) > 0:
                sim_block = self.item_sim_df.loc[valid].reindex(columns=self.all_movies, fill_value=0.0)
                cf_scores = (sim_block.T @ watched) / (watched.sum() + 1e-8)

        # Content Profile Component (TF-IDF Cosine Similarity)
        if seed_movie_id and self.content_sim_df is not None and seed_movie_id in self.content_sim_df.index:
            content_scores = self.content_sim_df.loc[seed_movie_id].reindex(self.all_movies, fill_value=0.0)
        elif not is_cold and user_id in self.user_profiles.index and self.tfidf_matrix is not None:
            u_vec = self.user_profiles.loc[user_id].values.reshape(1, -1)
            sims = cosine_similarity(u_vec, self.tfidf_matrix).flatten()
            content_scores = pd.Series(sims, index=self.movies_raw["movie_id"]).reindex(self.all_movies, fill_value=0.0)

        # Normalize Components
        norm_cf = self._normalize(cf_scores)
        norm_svd = self._normalize(svd_scores)
        norm_content = self._normalize(content_scores)
        norm_pop = self._normalize(pop_scores)

        # Weighted Blend Computation
        hybrid_scores = (
            self.w_cf * norm_cf +
            self.w_svd * norm_svd +
            self.w_content * norm_content +
            self.w_pop * norm_pop
        )

        # Exclude Already Watched / Seed Movies
        if not is_cold:
            watched_items = self.interaction_matrix.loc[user_id][self.interaction_matrix.loc[user_id] > 0].index
            hybrid_scores = hybrid_scores.drop(labels=watched_items, errors="ignore")

        if seed_movie_id:
            hybrid_scores = hybrid_scores.drop(labels=[seed_movie_id], errors="ignore")

        # Format Output Dataframe
        output = pd.DataFrame({
            "movie_id": hybrid_scores.index.to_numpy(),
            "cf_score": norm_cf.reindex(hybrid_scores.index).round(4).to_numpy(),
            "svd_score": norm_svd.reindex(hybrid_scores.index).round(4).to_numpy(),
            "content_score": norm_content.reindex(hybrid_scores.index).round(4).to_numpy(),
            "popularity_score": norm_pop.reindex(hybrid_scores.index).round(4).to_numpy(),
            "hybrid_score": hybrid_scores.round(4).to_numpy()}).reset_index(drop=True)

        cols_to_merge = ["movie_id", "title", "genre_primary"]
        if "imdb_rating" in self.movies_raw.columns:
            cols_to_merge.append("imdb_rating")

        output = output.merge(self.movies_raw[cols_to_merge], on="movie_id", how="left")

        return (
            output.sort_values("hybrid_score", ascending=False).head(top_n).reset_index(drop=True))


# Instantiate Advanced 4-Way SVD Recommender Engine
user_profiles_ref = user_profiles_df if "user_profiles_df" in globals() else user_profiles
content_sim_ref = content_sim_df if "content_sim_df" in globals() else None
tfidf_matrix_ref = tfidf_matrix if "tfidf_matrix" in globals() else None

advanced_engine = StreamIntelAdvancedRecommender(
    interaction_matrix=interaction_matrix,
    item_sim_df=item_similarity_df,
    svd_pred_df=svd_pred_df,
    user_profiles=user_profiles_ref,
    popularity_df=popularity_df,
    movies_df=movies_clean,
    content_sim_df=content_sim_ref,
    tfidf_matrix=tfidf_matrix_ref,
    weights=(0.15, 0.35, 0.35, 0.15)
)

# Serialize Advanced Engine Instance
advanced_artifact_path = MODELS_DIR / "advanced_svd_recommender_engine.joblib"
joblib.dump(advanced_engine, advanced_artifact_path, compress=3)

# Execution Audit Summary
print("ADVANCED 4-WAY HYBRID ENGINE INITIALIZED")
print(f"Weights (CF / SVD / Content / Pop) : ({advanced_engine.w_cf:.2f}, {advanced_engine.w_svd:.2f}, {advanced_engine.w_content:.2f}, {advanced_engine.w_pop:.2f})")
print(f"Indexed Movies / Profiled Users   : {len(advanced_engine.all_movies):,} / {len(advanced_engine.user_profiles):,}")

sample_user = interaction_matrix.index[0] if len(interaction_matrix) > 0 else "user_01741"
print(f"\nTEST RECOMMENDATIONS (User ID: {sample_user}):")
display(advanced_engine.predict(user_id=sample_user, top_n=5))

print(f"\nAdvanced Engine Serialized to: {advanced_artifact_path}")

ADVANCED 4-WAY HYBRID ENGINE INITIALIZED
Weights (CF / SVD / Content / Pop) : (0.15, 0.35, 0.35, 0.15)
Indexed Movies / Profiled Users   : 1,000 / 9,997

TEST RECOMMENDATIONS (User ID: user_00001):


,movie_id,cf_score,svd_score,content_score,popularity_score,hybrid_score,title,genre_primary,imdb_rating
0,movie_0200,0.0302,0.7804,0.9128,0.9291,0.7365,Legend Secret,Drama,4.6
1,movie_0313,0.0185,0.5922,0.9398,0.9370,0.6795,Quest Love,Adventure,0.5
2,movie_0254,0.0409,0.5986,0.8875,0.9302,0.6658,Empire Mystery,Drama,7.1
3,movie_0361,0.0110,0.4007,0.9893,0.9222,0.6265,War Adventure,Drama,6.8
4,movie_0195,0.0767,1.0000,0.3386,0.9488,0.6223,New Phoenix,Music,6.4



Advanced Engine Serialized to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/models/advanced_svd_recommender_engine.joblib


In [16]:
# 16. ADVANCED SVD HYBRID EVALUATION
# ---------------------------------------

# Ensure Target Directory Exists
MODELS_DIR.mkdir(parents=True, exist_ok=True)

K = 10
EVAL_SAMPLE_SIZE = 300

# Extract Test Users with Future Interactions
if "test_df" in globals() and not test_df.empty:
    test_users = test_df["user_id"].unique()
else:
    raise ValueError("`test_df` is not defined or is empty. Ensure train/test split was executed in earlier cells.")

# Sample evaluation users deterministically
np.random.seed(42)
eval_users = np.random.choice(
    test_users,
    size=min(EVAL_SAMPLE_SIZE, len(test_users)),
    replace=False)

user_precisions = []
user_recalls = []
user_aps = []
hit_counts = 0

# Iterative Ranking Evaluation Loop
for user_id in eval_users:
    # Ground truth items watched by the user in test set
    actual_items = set(test_df.loc[test_df["user_id"] == user_id, "movie_id"])

    if not actual_items:
        continue

    # Generate Top-K Recommendations from Advanced SVD Engine
    recs = advanced_engine.predict(
        user_id=user_id,
        top_n=K
    )

    if recs.empty or "movie_id" not in recs.columns:
        continue

    recommended_list = recs["movie_id"].tolist()

    # Binary relevance vector
    hits_list = [1 if item in actual_items else 0 for item in recommended_list]
    num_hits = sum(hits_list)

    if num_hits > 0:
        hit_counts += 1

    # Precision@K
    user_precisions.append(num_hits / K)

    # Recall@K
    user_recalls.append(num_hits / len(actual_items))

    # Average Precision@K (AP@K)
    running_hits = 0
    ap_score = 0.0

    for rank, hit in enumerate(hits_list, start=1):
        if hit:
            running_hits += 1
            ap_score += running_hits / rank

    if num_hits > 0:
        ap_score /= min(len(actual_items), K)

    user_aps.append(ap_score)


# Calculate Macro-Averaged Evaluation Metrics
precision_at_k = float(np.mean(user_precisions)) if user_precisions else 0.0
recall_at_k = float(np.mean(user_recalls)) if user_recalls else 0.0
map_at_k = float(np.mean(user_aps)) if user_aps else 0.0
hit_rate = float(hit_counts / len(eval_users)) if len(eval_users) > 0 else 0.0


# Extract Baseline Metric for Delta Comparison (if available)
baseline_map = None
if "best_row" in globals() and f"MAP@{K}" in best_row:
    baseline_map = float(best_row[f"MAP@{K}"])
elif (MODELS_DIR / "model_manifest.json").exists():
    try:
        with open(MODELS_DIR / "model_manifest.json", "r") as f:
            manifest_data = json.load(f)
            baseline_map = manifest_data.get("evaluation_metrics", {}).get("map_at_k")
    except Exception:
        baseline_map = None


# Save Evaluation Metrics JSON Artifact
svd_eval_metrics = {
    "model_name": "StreamIntelAdvancedRecommender",
    "evaluation_k": K,
    "eval_sample_size": len(eval_users),
    "precision_at_k": round(precision_at_k, 4),
    "recall_at_k": round(recall_at_k, 4),
    "map_at_k": round(map_at_k, 4),
    "hit_rate": round(hit_rate, 4),
    "weights_used": {
        "w_cf": advanced_engine.w_cf,
        "w_svd": advanced_engine.w_svd,
        "w_content": advanced_engine.w_content,
        "w_pop": advanced_engine.w_pop}}

eval_artifact_path = MODELS_DIR / "svd_hybrid_evaluation_metrics.json"
with open(eval_artifact_path, "w") as f:
    json.dump(svd_eval_metrics, f, indent=4)


# Execution & Audit Summary
print("\nADVANCED SVD HYBRID EVALUATION RESULTS")
print(f"Evaluated Users Count : {len(eval_users):,}")
print(f"Evaluation Target K   : {K}")
print(f"User Hit Rate (@{K})   : {hit_rate * 100:.2f}%")
print(f"Precision@{K:<2}          : {precision_at_k:.4f}")
print(f"Recall@{K:<2}             : {recall_at_k:.4f}")
print(f"MAP@{K:<2}                : {map_at_k:.4f}")

if baseline_map is not None:
    delta_map = map_at_k - baseline_map
    pct_change = (delta_map / baseline_map) * 100 if baseline_map > 0 else 0.0
    print(f"Baseline Engine MAP   : {baseline_map:.4f}")
    print(f"SVD Hybrid MAP Delta  : {delta_map:+.4f} ({pct_change:+.2f}%)")

print("\nSaved evaluation metrics to:", eval_artifact_path)


ADVANCED SVD HYBRID EVALUATION RESULTS
Evaluated Users Count : 300
Evaluation Target K   : 10
User Hit Rate (@10)   : 1.33%
Precision@10          : 0.0013
Recall@10             : 0.0089
MAP@10                : 0.0019
Baseline Engine MAP   : 0.0050
SVD Hybrid MAP Delta  : -0.0031 (-62.38%)

Saved evaluation metrics to: /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search/models/svd_hybrid_evaluation_metrics.json


In [19]:
# NOTEBOOK 03 — ARTIFACT ZIP BACKUP
# ---------------------------------------

from pathlib import Path
import shutil

ARTIFACT_ROOT = Path(
    "/kaggle/working/streamintel360_artifacts/notebook_03_recommendations"
)

zip_base = Path(
    "/kaggle/working/notebook_03_recommendations_backup"
)

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=ARTIFACT_ROOT
)

print("=" * 80)
print("NOTEBOOK 03 — ARTIFACT ZIP BACKUP")
print("=" * 80)

print(f"\nSource Folder : {ARTIFACT_ROOT}")
print(f"ZIP File      : {zip_path}")

zip_size_mb = Path(zip_path).stat().st_size / (1024 * 1024)

print(f"ZIP Size      : {zip_size_mb:.2f} MB")

print("\n✓ Notebook 03 artifacts successfully compressed.")

NOTEBOOK 03 + 04 — ARTIFACT ZIP BACKUP

Source Folder : /kaggle/working/streamintel360_artifacts/notebook_03_04_recommendations_search
ZIP File      : /kaggle/working/notebook_03_04_recommendations_search_backup.zip
ZIP Size      : 216.51 MB

✓ Notebook 03 + 04 artifacts successfully compressed.
✓ Download this ZIP to your laptop before the Kaggle session ends.
